In [2]:
from ultralytics import SAM, FastSAM
from segmentation import start_segmentation
import torch
import numpy as np
import cv2

In [3]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
# Load a model
model = SAM("sam_b.pt")
# Display model information (optional)
model.info()

# Load a model
model = FastSAM("FastSAM-s.pt")
# Display model information (optional)
model.info()

C:\Users\bocca\anaconda3\Lib\site-packages\ultralytics\models\sam\build.py:134: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


Model summary: 238 layers, 93735472 parameters, 93735472 gradients


C:\Users\bocca\anaconda3\Lib\site-packages\ultralytics\nn\tasks.py:732: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(file, map_location="cpu")


YOLOv8s-seg summary: 261 layers, 11790483 parameters, 0 gradients, 42.7 GFLOPs


(261, 11790483, 0, 42.690713599999995)

In [9]:
def random_palette(n):
    rng = np.random.default_rng(42)   # reproductible
    return rng.integers(0, 256, size=(n, 3), dtype=np.uint8)

PALETTE = np.random.default_rng(42).integers(0, 256, size=(255, 3), dtype=np.uint8)
next_id = 0
tracks = {}      # {id: (cx, cy)}
def match(centroid, tracks, thresh=40):
    """Retourne l’id existant proche (< thresh) ou None."""
    cx, cy = centroid
    for tid, (tx, ty) in tracks.items():
        if (cx - tx)**2 + (cy - ty)**2 < thresh**2:
            return tid
    return None

def segmenter(frame, fast=False, with_frame=False):
    global next_id
    global tracks
    # Load a model
    if fast:
        model = FastSAM("FastSAM-s.pt").to(device)
    else:
        model = SAM("sam_b.pt").to(device)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = model(rgb)
    if with_frame:
        H, W = rgb.shape[:2]
        blank = np.zeros((H, W, 3), dtype=np.uint8)
        annotated_rgb = results[0].plot(
            img=frame,
            masks=True,      # garde les masques
            boxes=False,     # supprime les bounding-boxes
            labels=False     # supprime les libellés
        )
        annotated_bgr = cv2.cvtColor(annotated_rgb, cv2.COLOR_RGB2BGR)
        return annotated_bgr
    else:
        masks = results[0].masks.data.cpu().numpy()  # (N, H, W) – booléen/float
        N, H, W = masks.shape
        palette = random_palette(N)                  # (N, 3)
    
        canvas = np.zeros((H, W, 3), dtype=np.uint8) # fond noir
    
        new_tracks = {}
        for mask in masks:
            ys, xs = np.where(mask)
            if len(xs) == 0:            # masque vide
                continue
            cx, cy = xs.mean(), ys.mean()
            tid = match((cx, cy), tracks)
    
            if tid is None:             # nouvel objet
                tid = next_id
                next_id += 1
            new_tracks[tid] = (cx, cy)
    
            color = PALETTE[tid % len(PALETTE)]
            canvas[mask > 0] = color
    
        tracks = new_tracks
        
        cv2.imshow("initial", frame)# on met à jour pour la frame suivante
        return canvas

In [10]:
start_segmentation(segmenter=segmenter)

C:\Users\bocca\anaconda3\Lib\site-packages\ultralytics\models\sam\build.py:134: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)



0: 1024x1024 1 0, 1 1, 1 2, 1 3, 1 4, 1 5, 1 6, 1 7, 1 8, 1 9, 1 10, 1 11, 1 12, 1 13, 1 14, 1 15, 1 16, 1 17, 1 18, 1 19, 1 20, 1 21, 1 22, 1 23, 1 24, 1 25, 1 26, 1 27, 1 28, 1 29, 1 30, 1 31, 1 32, 1 33, 1 34, 1 35, 1 36, 1 37, 1 38, 1 39, 1 40, 1 41, 1 42, 1 43, 1 44, 1 45, 1 46, 1 47, 1 48, 1 49, 1 50, 1 51, 1 52, 1 53, 1 54, 1 55, 1 56, 1 57, 1 58, 1 59, 1 60, 1 61, 84862.0ms
Speed: 46.5ms preprocess, 84862.0ms inference, 10.5ms postprocess per image at shape (1, 3, 1024, 1024)

0: 1024x1024 1 0, 1 1, 1 2, 1 3, 1 4, 1 5, 1 6, 1 7, 1 8, 1 9, 1 10, 1 11, 1 12, 1 13, 1 14, 1 15, 1 16, 1 17, 1 18, 1 19, 1 20, 1 21, 1 22, 1 23, 1 24, 1 25, 1 26, 1 27, 1 28, 1 29, 1 30, 1 31, 1 32, 1 33, 1 34, 1 35, 1 36, 1 37, 1 38, 1 39, 1 40, 1 41, 1 42, 1 43, 1 44, 1 45, 1 46, 1 47, 1 48, 1 49, 1 50, 1 51, 1 52, 1 53, 1 54, 1 55, 1 56, 1 57, 1 58, 1 59, 1 60, 1 61, 1 62, 1 63, 1 64, 1 65, 1 66, 1 67, 1 68, 86626.7ms
Speed: 13.5ms preprocess, 86626.7ms inference, 7.0ms postprocess per image at shap

KeyboardInterrupt: 